In [ ]:
#PCA Anomaly
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from sklearn.decomposition import PCA

# ── Configurações ──────────────────────────────────────────────
REAL_PATH = "foto_real.jpg"
AI_PATH   = "foto_ia.jpg"
RESIZE_TO = (128, 128)
N_COMPONENTS = 30
# ──────────────────────────────────────────────────────────────

def load_image(path):
    img_rgb  = Image.open(path).convert('RGB').resize(RESIZE_TO)
    img_gray = img_rgb.convert('L')
    arr_rgb  = np.array(img_rgb,  dtype=np.float64) / 255.0
    arr_gray = np.array(img_gray, dtype=np.float64) / 255.0
    return img_rgb, arr_rgb, arr_gray

def pca_reconstruction(arr_gray, k):
    n   = min(N_COMPONENTS, min(arr_gray.shape))
    pca = PCA(n_components=n, svd_solver='randomized', random_state=42)
    scores = pca.fit_transform(arr_gray)
    sk = np.zeros_like(scores)
    sk[:, :k] = scores[:, :k]
    return pca.inverse_transform(sk), pca.explained_variance_ratio_

def fft_spectrum(arr_gray):
    fft     = np.fft.fft2(arr_gray)
    fft_mag = np.abs(np.fft.fftshift(fft))
    return np.log1p(fft_mag)

def fft_energy_bars(arr_gray):
    fft     = np.fft.fft2(arr_gray)
    fft_mag = np.abs(np.fft.fftshift(fft))
    h, w    = fft_mag.shape
    cy, cx  = h // 2, w // 2
    Y, X    = np.ogrid[:h, :w]
    dist    = np.sqrt((X - cx)**2 + (Y - cy)**2)
    max_d   = np.sqrt(cx**2 + cy**2)

    e_low   = float(np.sum(fft_mag[dist <= max_d * 0.1]**2))
    e_mid   = float(np.sum(fft_mag[(dist > max_d*0.1) & (dist <= max_d*0.4)]**2))
    e_high  = float(np.sum(fft_mag[dist > max_d * 0.4]**2))
    total   = e_low + e_mid + e_high + 1e-12
    return [e_low/total, e_mid/total, e_high/total]

def color_histograms(arr_rgb):
    histograms = []
    for c in range(3):
        hist, bins = np.histogram(arr_rgb[:,:,c].ravel(), bins=64, range=(0,1))
        histograms.append((hist, bins))
    return histograms

def compare_full(real_path, ai_path):
    img_r, arr_r, gray_r = load_image(real_path)
    img_a, arr_a, gray_a = load_image(ai_path)

    # PCA
    rec_r5,  evr_r = pca_reconstruction(gray_r, 5)
    rec_a5,  evr_a = pca_reconstruction(gray_a, 5)
    rec_r20, _     = pca_reconstruction(gray_r, 20)
    rec_a20, _     = pca_reconstruction(gray_a, 20)

    # FFT
    fft_r = fft_spectrum(gray_r)
    fft_a = fft_spectrum(gray_a)
    bars_r = fft_energy_bars(gray_r)
    bars_a = fft_energy_bars(gray_a)

    # Histogramas RGB
    hist_r = color_histograms(arr_r)
    hist_a = color_histograms(arr_a)

    # ── Layout ────────────────────────────────────────────────
    fig = plt.figure(figsize=(18, 22), facecolor='#0f0f0f')
    fig.suptitle("Análise de Features — Real vs IA", fontsize=18,
                 fontweight='bold', color='white', y=0.98)

    gs = gridspec.GridSpec(5, 4, figure=fig, hspace=0.45, wspace=0.35)

    COR_R = '#4ade80'   # verde
    COR_A = '#60a5fa'   # azul

    def ax(row, col, colspan=1):
        return fig.add_subplot(gs[row, col:col+colspan])

    def title(a, t, cor):
        a.set_title(t, color=cor, fontsize=10, pad=4)

    def style(a):
        a.set_facecolor('#1a1a1a')
        a.tick_params(colors='#888', labelsize=7)
        for sp in a.spines.values():
            sp.set_edgecolor('#333')

    # ── Linha 0: imagens originais ────────────────────────────
    a = ax(0, 0, 2)
    a.imshow(np.array(img_r))
    title(a, "Real — RGB original", COR_R); a.axis('off')

    a = ax(0, 2, 2)
    a.imshow(np.array(img_a))
    title(a, "IA — RGB original", COR_A); a.axis('off')

    # ── Linha 1: PCA reconstrução k=5 e k=20 ─────────────────
    for col, rec5, rec20, evr, cor, label in [
        (0, rec_r5, rec_r20, evr_r, COR_R, "Real"),
        (2, rec_a5, rec_a20, evr_a, COR_A, "IA"),
    ]:
        a = ax(1, col)
        a.imshow(np.clip(rec5, 0, 1), cmap='gray')
        title(a, f"{label} — PCA k=5 (reconstrução)", cor); a.axis('off')

        a = ax(1, col+1)
        a.imshow(np.clip(rec20, 0, 1), cmap='gray')
        title(a, f"{label} — PCA k=20 (reconstrução)", cor); a.axis('off')

    # ── Linha 2: Variância explicada PCA ─────────────────────
    for col, evr, cor, label in [
        (0, evr_r, COR_R, "Real"), (2, evr_a, COR_A, "IA")
    ]:
        a = ax(2, col, 2); style(a)
        n = len(evr)
        a.bar(range(n), evr * 100, color=cor, alpha=0.85, width=0.8)
        a.set_xlabel("Componente PCA", color='#888', fontsize=8)
        a.set_ylabel("Variância explicada (%)", color='#888', fontsize=8)
        title(a, f"{label} — Variância por componente PCA", cor)

    # ── Linha 3: Espectro FFT ─────────────────────────────────
    for col, fft, bars, cor, label in [
        (0, fft_r, bars_r, COR_R, "Real"),
        (2, fft_a, bars_a, COR_A, "IA"),
    ]:
        a = ax(3, col)
        a.imshow(fft, cmap='magma'); a.axis('off')
        title(a, f"{label} — Espectro FFT (log)", cor)

        a = ax(3, col+1); style(a)
        cores_bar = ['#facc15', '#fb923c', '#f87171']
        a.bar(['Baixa\nfreq', 'Média\nfreq', 'Alta\nfreq'],
              [b*100 for b in bars], color=cores_bar, alpha=0.9)
        a.set_ylabel("Energia (%)", color='#888', fontsize=8)
        title(a, f"{label} — Distribuição de energia FFT", cor)

    # ── Linha 4: Histogramas RGB ──────────────────────────────
    canal_cores = ['#f87171', '#4ade80', '#60a5fa']  # R, G, B
    canal_nomes = ['R', 'G', 'B']

    for col_offset, hist_data, cor, label in [
        (0, hist_r, COR_R, "Real"),
        (2, hist_a, COR_A, "IA"),
    ]:
        a = ax(4, col_offset, 2); style(a)
        for (h, bins), cc, cn in zip(hist_data, canal_cores, canal_nomes):
            centers = (bins[:-1] + bins[1:]) / 2
            a.plot(centers, h, color=cc, alpha=0.85, linewidth=1.5, label=cn)
        a.legend(fontsize=8, facecolor='#1a1a1a', labelcolor='white',
                 edgecolor='#333')
        a.set_xlabel("Intensidade", color='#888', fontsize=8)
        a.set_ylabel("Contagem de pixels", color='#888', fontsize=8)
        title(a, f"{label} — Histograma RGB", cor)

    plt.savefig("analise_features.png", dpi=150, bbox_inches='tight',
                facecolor='#0f0f0f')
    plt.show()
    print("Salvo: analise_features.png")



REAL_PATH = "descubra-agora-5-cuidados-que-se-deve-ter-com-os-filhotes-de-cachorro-3.jpg"
AI_PATH   = "ChatGPT Image 26 de mai. de 2026, 13_53_33.png"
# ────────────────────────────────────────

compare_full(REAL_PATH, AI_PATH)

In [ ]:
#Ruido
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from scipy.ndimage import convolve

# ── Configurações ──────────────────────────────────────────────
REAL_PATH  = "foto_real.jpg"
AI_PATH    = "foto_ia.jpg"
RESIZE_TO  = (256, 256)
PATCH_SIZE = 32
N_PATCHES  = 3
# ──────────────────────────────────────────────────────────────

SRM_FILTERS = [
    np.array([[0, 0, 0], [0, -1, 1], [0, 0, 0]], dtype=np.float32),
    np.array([[0, 0, 0], [0, -1, 0], [0, 1, 0]], dtype=np.float32),
    np.array([[0, 0, 0], [0, -2, 1], [0, 1, 0]], dtype=np.float32),
    np.array([[-1, 2, -1], [0, 0, 0], [0, 0, 0]], dtype=np.float32),
    np.array([[0, 0, 0], [-1, 2, -1], [0, 0, 0]], dtype=np.float32),
    np.array([[-1, 0, 0], [2, 0, 0], [-1, 0, 0]], dtype=np.float32),
    np.array([[0, -1, 0], [0, 2, 0], [0, -1, 0]], dtype=np.float32),
    np.array([[-1, 0, 1], [0, 0, 0], [1, 0, -1]], dtype=np.float32),
    np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32),
]

def load_gray(path):
    img = Image.open(path).convert("L").resize(RESIZE_TO)
    arr = np.array(img, dtype=np.float32) / 255.0
    return img, arr

def dct_score(patch):
    return float(np.sum(np.abs(cv2.dct(patch.astype(np.float32)))))

def get_patches(arr):
    h, w = arr.shape
    patches = []
    for y in range(0, h - PATCH_SIZE + 1, PATCH_SIZE):
        for x in range(0, w - PATCH_SIZE + 1, PATCH_SIZE):
            patch = arr[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            patches.append((dct_score(patch), patch, y, x))
    patches.sort(key=lambda p: p[0])
    return patches

def srm_noise_map(arr, filter_idx=8):
    """Aplica um filtro SRM na imagem inteira para visualizar o ruído."""
    f = SRM_FILTERS[filter_idx]
    return convolve(arr.astype(np.float32), f)

def laplacian_map(arr):
    lap = cv2.Laplacian((arr * 255).astype(np.uint8), cv2.CV_64F)
    return lap

def dct_heatmap(arr):
    """Gera heatmap de scores DCT por patch."""
    h, w   = arr.shape
    hmap   = np.zeros((h // PATCH_SIZE, w // PATCH_SIZE), dtype=np.float32)
    for yi, y in enumerate(range(0, h - PATCH_SIZE + 1, PATCH_SIZE)):
        for xi, x in enumerate(range(0, w - PATCH_SIZE + 1, PATCH_SIZE)):
            patch = arr[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            hmap[yi, xi] = dct_score(patch)
    return hmap

def highlight_patches(arr, patches, n, color_low, color_high):
    """Desenha retângulos nos patches de menor e maior frequência."""
    vis = cv2.cvtColor((arr * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    for _, _, y, x in patches[:n]:      # baixa frequência
        cv2.rectangle(vis, (x, y), (x+PATCH_SIZE, y+PATCH_SIZE), color_low,  2)
    for _, _, y, x in patches[-n:]:     # alta frequência
        cv2.rectangle(vis, (x, y), (x+PATCH_SIZE, y+PATCH_SIZE), color_high, 2)
    return vis

def compare(real_path, ai_path):
    img_r, arr_r = load_gray(real_path)
    img_a, arr_a = load_gray(ai_path)

    patches_r = get_patches(arr_r)
    patches_a = get_patches(arr_a)

    noise_r   = srm_noise_map(arr_r)
    noise_a   = srm_noise_map(arr_a)
    lap_r     = laplacian_map(arr_r)
    lap_a     = laplacian_map(arr_a)
    hmap_r    = dct_heatmap(arr_r)
    hmap_a    = dct_heatmap(arr_a)
    patch_vis_r = highlight_patches(arr_r, patches_r, N_PATCHES,
                                    color_low=(255,200,0), color_high=(0,200,255))
    patch_vis_a = highlight_patches(arr_a, patches_a, N_PATCHES,
                                    color_low=(255,200,0), color_high=(0,200,255))

    # Estatísticas Laplaciano
    stats = {
        "Variância":    [float(np.var(lap_r)),                       float(np.var(lap_a))],
        "Média |L|":    [float(np.mean(np.abs(lap_r))),              float(np.mean(np.abs(lap_a)))],
        "Desvio Padrão":[float(np.std(lap_r)),                       float(np.std(lap_a))],
        "Percentil 90": [float(np.percentile(np.abs(lap_r), 90)),    float(np.percentile(np.abs(lap_a), 90))],
    }

    # ── Layout ────────────────────────────────────────────────
    fig = plt.figure(figsize=(18, 24), facecolor='#0f0f0f')
    fig.suptitle("Análise de Ruído e Frequência — Real vs IA\n"
                 "(DCT · Filtros SRM · Laplaciano)",
                 fontsize=17, fontweight='bold', color='white', y=0.99)

    gs  = gridspec.GridSpec(5, 4, figure=fig, hspace=0.45, wspace=0.35)
    CR  = '#4ade80'
    CA  = '#60a5fa'

    def ax(r, c, cs=1): return fig.add_subplot(gs[r, c:c+cs])
    def style(a):
        a.set_facecolor('#1a1a1a')
        a.tick_params(colors='#888', labelsize=7)
        for sp in a.spines.values(): sp.set_edgecolor('#333')
    def tit(a, t, c): a.set_title(t, color=c, fontsize=10, pad=4)

    # Linha 0 — Imagens originais em cinza
    a = ax(0, 0, 2); a.imshow(arr_r, cmap='gray'); a.axis('off')
    tit(a, "Real — Escala de Cinza (entrada do algoritmo)", CR)
    a = ax(0, 2, 2); a.imshow(arr_a, cmap='gray'); a.axis('off')
    tit(a, "IA — Escala de Cinza (entrada do algoritmo)", CA)

    # Linha 1 — Seleção de patches por DCT
    a = ax(1, 0, 2); a.imshow(patch_vis_r); a.axis('off')
    tit(a, "Real — Patches selecionados\n[A] baixa freq  [B] alta freq", CR)
    a = ax(1, 2, 2); a.imshow(patch_vis_a); a.axis('off')
    tit(a, "IA — Patches selecionados\n[A] baixa freq  [B] alta freq", CA)

    # Linha 2 — Heatmap DCT por região
    a = ax(2, 0, 2); style(a)
    im = a.imshow(hmap_r, cmap='inferno', aspect='auto')
    plt.colorbar(im, ax=a, fraction=0.03, pad=0.02).ax.tick_params(colors='#888')
    tit(a, "Real — Heatmap de Frequência DCT por Patch", CR)

    a = ax(2, 2, 2); style(a)
    im = a.imshow(hmap_a, cmap='inferno', aspect='auto')
    plt.colorbar(im, ax=a, fraction=0.03, pad=0.02).ax.tick_params(colors='#888')
    tit(a, "IA — Heatmap de Frequência DCT por Patch", CA)

    # Linha 3 — Mapa de ruído SRM (filtro Laplaciano 2D)
    noise_r_norm = (noise_r - noise_r.min()) / (noise_r.max() - noise_r.min() + 1e-8)
    noise_a_norm = (noise_a - noise_a.min()) / (noise_a.max() - noise_a.min() + 1e-8)   

    a = ax(3, 0, 2); a.imshow(noise_r_norm, cmap='hot'); a.axis('off')
    tit(a, "Real — Mapa de Ruído SRM (filtro Laplaciano 2D)", CR)
    a = ax(3, 2, 2); a.imshow(noise_a_norm, cmap='hot'); a.axis('off')
    tit(a, "IA — Mapa de Ruído SRM (filtro Laplaciano 2D)", CA)

    # Linha 4 — Estatísticas do Laplaciano (barras lado a lado)
    a = ax(4, 0, 4); style(a)
    labels   = list(stats.keys())
    vals_r   = [stats[k][0] for k in labels]
    vals_a   = [stats[k][1] for k in labels]
    x        = np.arange(len(labels))
    w        = 0.35
    bars_r   = a.bar(x - w/2, vals_r, w, color=CR, alpha=0.85, label='Real')
    bars_a   = a.bar(x + w/2, vals_a, w, color=CA, alpha=0.85, label='IA')
    a.set_xticks(x); a.set_xticklabels(labels, color='#ccc', fontsize=10)
    a.set_ylabel("Valor", color='#888', fontsize=9)
    a.legend(fontsize=10, facecolor='#1a1a1a', labelcolor='white', edgecolor='#333')
    tit(a, "Estatísticas do Laplaciano — Ruído Global da Imagem\n"
           "(valores maiores = mais ruído natural = mais provável ser Real)", 'white')

    # Anotações nas barras
    for bar in bars_r:
        a.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
               f'{bar.get_height():.1f}', ha='center', va='bottom',
               color=CR, fontsize=8)
    for bar in bars_a:
        a.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
               f'{bar.get_height():.1f}', ha='center', va='bottom',
               color=CA, fontsize=8)

    plt.savefig("analise_ruido.png", dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
    print("Salvo: analise_ruido.png")

REAL_PATH = "descubra-agora-5-cuidados-que-se-deve-ter-com-os-filhotes-de-cachorro-3.jpg"
AI_PATH   = "ChatGPT Image 26 de mai. de 2026, 13_53_33.png"

compare(REAL_PATH, AI_PATH)